# DPA Algorithm Walkthrough

A demonstration of DPA clustering on **real data** (UCI Optdigits).

This notebook shows:
1. Intrinsic dimension estimation (TWO-NN with Block Analysis)
2. Adaptive density estimation (PAk)
3. Automatic cluster detection
4. Topographic visualization

For detailed parameter study see `02_parameter_study.ipynb`, for comparisons with other methods see `03_comparison_analysis.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import sys
sys.path.insert(0, '..')

from src import DPA, TwoNN, PAk, load_optdigits
from src.intrinsic_dimension import BlockAnalysis

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
# Load real data: UCI Optdigits (8x8 handwritten digits)
data = load_optdigits()
X = data['data']
y_true = data['target']

print(f"Dataset: Optdigits (UCI)")
print(f"  Samples: {X.shape[0]}")
print(f"  Features: {X.shape[1]} (8x8 pixels)")
print(f"  Classes: {data['n_classes']} (digits 0-9)")

# Show sample digits
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    idx = np.where(y_true == i)[0][0]
    ax.imshow(X[idx].reshape(8, 8), cmap='gray')
    ax.set_title(f'Digit {i}')
    ax.axis('off')
plt.suptitle('Sample digits from Optdigits dataset')
plt.tight_layout()
plt.show()

## 1. Intrinsic Dimension Estimation

The TWO-NN method estimates the intrinsic dimension (d) of the data manifold.
- Optdigits has 64 features but lies on a lower-dimensional manifold
- Paper reports d ~ 7-9 for similar digit data
- Block Analysis shows how d varies with sample size (scale)

In [ ]:
# Block Analysis - how dimension estimate changes with scale
ba = BlockAnalysis(n_blocks=10, min_samples=100, n_repetitions=5, random_state=42)
ba.fit(X)

print(ba.interpret())
print(f"\nFull dataset TWO-NN estimate: d = {TwoNN().fit(X).dimension_:.2f}")

ba.plot()
plt.title('Block Analysis: Intrinsic Dimension vs Scale')
plt.show()

## 2. DPA Clustering

DPA automatically detects clusters using:
- PAk adaptive density estimation
- Peak detection with error bounds
- Z-score based peak merging (Z parameter controls confidence level)
- Halo point identification for uncertain assignments

In [ ]:
# Fit DPA
# Using Z=1.5 for high-dimensional data (larger errors -> need lower Z)
dpa = DPA(Z=1.5, halo=True)
labels = dpa.fit_predict(X)

# Summary
print("DPA Results:")
print("-" * 40)
summary = dpa.summary()
print(f"  Intrinsic dimension: {summary['intrinsic_dimension']:.2f}")
print(f"  Clusters found: {summary['n_clusters']} (ground truth: 10)")
print(f"  Halo points: {summary['n_halo_points']}")
print(f"  k_hat range: [{summary['k_hat_stats']['min']}, {summary['k_hat_stats']['max']}]")
print(f"  k_hat mean: {summary['k_hat_stats']['mean']:.1f}")

In [ ]:
# Visualize results using PCA for 2D projection
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)
print(f"PCA variance explained: {pca.explained_variance_ratio_.sum():.1%}")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Ground truth
ax = axes[0, 0]
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_true, cmap='tab10', alpha=0.6, s=15)
ax.set_title(f'Ground Truth (10 digit classes)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')

# DPA Clustering
ax = axes[0, 1]
core_mask = labels >= 0
scatter = ax.scatter(X_2d[core_mask, 0], X_2d[core_mask, 1], 
                     c=labels[core_mask], cmap='tab20', alpha=0.6, s=15)
if np.any(~core_mask):
    ax.scatter(X_2d[~core_mask, 0], X_2d[~core_mask, 1], 
               c='gray', alpha=0.3, s=5, label=f'Halo ({np.sum(~core_mask)})')
    ax.legend(fontsize=8)
# Mark cluster centers
centers_2d = X_2d[dpa.cluster_centers_]
ax.scatter(centers_2d[:, 0], centers_2d[:, 1], 
           c='red', s=100, marker='*', edgecolors='black', linewidths=1)
ax.set_title(f'DPA Clustering ({dpa.n_clusters_} clusters, Z={dpa.Z})')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')

# Density landscape
ax = axes[1, 0]
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=dpa.log_density_, 
                     cmap='viridis', alpha=0.6, s=15)
plt.colorbar(scatter, ax=ax, label='log(rho)')
ax.set_title('Density Landscape (PAk estimate)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')

# Decision graph
ax = axes[1, 1]
dpa.plot_topography(kind='decision', ax=ax)
ax.set_title('Decision Graph')

plt.tight_layout()
plt.show()

In [ ]:
# Evaluation metrics
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(y_true, dpa.labels_full_)
nmi = normalized_mutual_info_score(y_true, dpa.labels_full_)

print("Clustering Quality:")
print("-" * 40)
print(f"  Adjusted Rand Index (ARI): {ari:.3f}")
print(f"  Normalized Mutual Info (NMI): {nmi:.3f}")
print()
print("Note: DPA may find more clusters than ground truth classes")
print("because it discovers natural density peaks, which may include")
print("sub-classes within digits (e.g., different writing styles).")